# Hyperparameter tuning in practice

In the previous notebook, we used a fixed validation set to make model selection visible. In practice, we usually use **cross-validation within the training data** to compare hyperparameter settings, while keeping the test set untouched until the end.

This notebook continues with the same noisy spiral classification problem and shows the standard scikit-learn workflow.

### Learning goals

After this notebook, you should be able to:

- tune several hyperparameters with `GridSearchCV`;
- explain what cross-validation is doing inside the search;
- use `RandomizedSearchCV` when exhaustive search becomes expensive;
- compare grid and randomized search in terms of search space and computational budget;
- evaluate the selected model once on an untouched test set;
- explain when a more advanced optimizer such as Optuna can be useful.


## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import randint
from sklearn import metrics, tree
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)

# Find data/spiral_noise.csv by searching upward from the notebook directory.
project_root = Path.cwd().resolve()
while project_root.parent != project_root:
    if (project_root / "data" / "spiral_noise.csv").is_file():
        break
    project_root = project_root.parent

data_path = project_root / "data" / "spiral_noise.csv"

if not data_path.is_file():
    raise FileNotFoundError(
        "Could not find data/spiral_noise.csv. "
        "Run this notebook from inside the ML book project."
    )

## Prepare the same noisy spiral data

We use the same setup as in the previous notebook: 1000 observations and 10% deliberately flipped labels.

This time we make only one split:

$$
\text{labeled data}
\longrightarrow
\begin{cases}
\text{training set}\\
\text{test set}
\end{cases}
$$

We do **not** create a separate validation set. `GridSearchCV` and `RandomizedSearchCV` will create validation folds internally from the training set.

In [2]:
df = pd.read_csv(data_path)

feature_cols = ["A", "B"]
target_var = "Target"

# Match the data preparation used in the conceptual notebook.
df = df.sample(frac=1, random_state=70).reset_index(drop=True)
df = df.iloc[:1000].copy()

X = df[feature_cols].to_numpy()
y = df[target_var].to_numpy().copy()

# Add the same amount of label noise.
noise = 0.10

rng = np.random.default_rng(42)
n_flip = int(noise * len(y))
flip_idx = rng.choice(len(y), size=n_flip, replace=False)
y[flip_idx] = 1 - y[flip_idx]

# Hold out the test set before any hyperparameter tuning.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Complete labeled dataset: {len(X):4d}")
print(f"├── Training set:         {len(X_train):4d}")
print(f"└── Test set:             {len(X_test):4d}")

Complete labeled dataset: 1000
├── Training set:          800
└── Test set:              200


**Check.** Which data may be used to choose hyperparameters?

<details>
<summary><strong>Answer</strong></summary>

Only the training data. Cross-validation will repeatedly divide the training set into fitting and validation folds. The test set remains untouched until model selection is complete.

</details>

**Takeaway.** Cross-validation replaces one fixed validation split with several validation folds, but it does not replace the final test set.

## Grid search with cross-validation

Suppose we want to tune two aspects of a decision tree:

- `max_leaf_nodes`: how many terminal regions the tree may create;
- `min_samples_leaf`: the minimum number of training examples allowed in a leaf.

`GridSearchCV` evaluates **every combination** in a specified grid. Each combination is evaluated using cross-validation.

We use stratified 5-fold cross-validation so that each fold has approximately the same class proportions as the complete training set.

In [3]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

param_grid = {
    "max_leaf_nodes": [5, 10, 20, 40, 80],
    "min_samples_leaf": [1, 2, 5, 10],
}

grid_search = GridSearchCV(
    estimator=tree.DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

**Predict.** How many candidate hyperparameter combinations will the grid contain, and how many cross-validation fits will this require?

<details>
<summary><strong>Answer</strong></summary>

There are $5 \times 4 = 20$ hyperparameter combinations. With 5-fold cross-validation, this requires $20 \times 5 = 100$ cross-validation fits. After the search, scikit-learn also refits the selected configuration on the complete training set.

</details>

In [4]:
grid_search.fit(X_train, y_train)

grid_results = pd.DataFrame(grid_search.cv_results_)

print(f"Candidate configurations: {len(grid_results)}")
print(f"Cross-validation fits:    {len(grid_results) * cv.get_n_splits()}")
print()
print("Best parameters:", grid_search.best_params_)
print(f"Best mean CV accuracy: {grid_search.best_score_:.3f}")

Candidate configurations: 20
Cross-validation fits:    100

Best parameters: {'max_leaf_nodes': 40, 'min_samples_leaf': 10}
Best mean CV accuracy: 0.766


The search object stores the result of every candidate configuration in `cv_results_`. Looking at these results is useful: hyperparameter search should not be treated as a black box.

In [5]:
(
    grid_results[
        [
            "param_max_leaf_nodes",
            "param_min_samples_leaf",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .sort_values("rank_test_score")
    .head(10)
)

,param_max_leaf_nodes,param_min_samples_leaf,mean_test_score,std_test_score,rank_test_score
19,80,10,0.76625,0.018792,1
15,40,10,0.76625,0.018792,1
11,20,10,0.76625,0.011592,3
10,20,5,0.75875,0.009354,4
8,20,1,0.75750,0.016489,5
9,20,2,0.75750,0.016489,5
16,80,1,0.75250,0.016105,7
12,40,1,0.74875,0.028339,8
13,40,2,0.74875,0.024812,8
14,40,5,0.73875,0.016008,10


**Think.** Why is `mean_test_score` in `cv_results_` not the performance on our model on new data?

<details>
<summary><strong>Answer</strong></summary>

In scikit-learn's cross-validation terminology, each validation fold is called a *test fold*. Here, `mean_test_score` is therefore the mean score on the cross-validation validation folds. Our held-out `X_test` has still not been used.

</details>

**Takeaway.** `GridSearchCV` uses cross-validation to compare every configuration in the grid; the independent test set remains outside the search.

## When grid search becomes expensive

Grid search is straightforward when the search space is small. Its cost grows rapidly, however, because it evaluates every combination.

If five hyperparameters each have ten candidate values, the grid contains

$$
10^5 = 100\,000
$$

configurations. With 5-fold cross-validation, evaluating the complete grid requires 500,000 cross-validation fits.

Often we would rather specify how much computation we can afford than evaluate every point on a large grid.

## Randomized search

`RandomizedSearchCV` samples a specified number of hyperparameter combinations instead of evaluating the complete search space.

This lets us control the computational budget directly with `n_iter`. It is particularly useful when there are several hyperparameters or wide ranges of plausible values.

For integer-valued hyperparameters, we can describe ranges using probability distributions rather than listing every possible value.

In [6]:
param_distributions = {
    "max_leaf_nodes": randint(5, 101),
    "min_samples_leaf": randint(1, 21),
    "max_depth": randint(2, 21),
}

random_search = RandomizedSearchCV(
    estimator=tree.DecisionTreeClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=30,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
)

random_search.fit(X_train, y_train)

print(f"Candidate configurations: {len(random_search.cv_results_['params'])}")
print(f"Cross-validation fits:    {len(random_search.cv_results_['params']) * cv.get_n_splits()}")
print()
print("Best parameters:", random_search.best_params_)
print(f"Best mean CV accuracy: {random_search.best_score_:.3f}")

Candidate configurations: 30
Cross-validation fits:    150

Best parameters: {'max_depth': 20, 'max_leaf_nodes': 91, 'min_samples_leaf': 11}
Best mean CV accuracy: 0.772


**Think.** Is randomized search inherently better than grid search?

<details>
<summary><strong>Answer</strong></summary>

No. Grid search is simple and effective when the search space is small. Randomized search is often more practical when the space is large because it can explore wider ranges while evaluating only a fixed number of configurations.

</details>

A useful distinction is:

- **Grid search:** evaluate every combination in a predefined grid.
- **Randomized search:** sample a fixed number of combinations from a search space.

**Takeaway.** Randomized search is mainly an efficiency strategy: it spends a limited search budget across a larger space.

## Beyond random search: Optuna

Grid and randomized search do not use the results of earlier trials to decide which hyperparameters to try next. More advanced optimization methods can use previous results to concentrate the search on promising parts of the hyperparameter space.

One widely used library for this is **Optuna**. It is a separate Python package, so it must be installed before use:

```bash
pip install optuna
```

or, when using conda:

```bash
conda install -c conda-forge optuna
```

With Optuna, we define an **objective function** and ranges from which hyperparameters may be chosen. The optimizer uses results from previous trials to guide later trials toward promising regions. It also supports **pruning**, which can stop unpromising trials early when the learning algorithm provides intermediate results.

This can make Optuna more efficient than an exhaustive grid, especially when model fitting is expensive or the hyperparameter space is large. It is not automatically better for every problem: for a small search space such as our decision-tree example, `GridSearchCV` is simpler and entirely adequate.

The important idea is the progression:

$$
\text{grid search: exhaustive}
\quad\longrightarrow\quad
\text{random search: fixed budget}
\quad\longrightarrow\quad
\text{guided optimization: learn from previous trials}.
$$

We will not use the Optuna API here; the goal is to recognize when a more advanced search strategy may be useful.

## Final evaluation on the test set

Only now, after the search is complete, do we use the test set.

By default, both `GridSearchCV` and `RandomizedSearchCV` use `refit=True`. After selecting the best hyperparameters, scikit-learn automatically fits that configuration once more on the **complete training set**. The resulting model is available as `best_estimator_`.

For the final model below, we use the result of the grid search. The choice of search procedure itself should not be made by comparing their test-set scores.

In [7]:
best_model = grid_search.best_estimator_

test_predictions = best_model.predict(X_test)
test_accuracy = metrics.accuracy_score(y_test, test_predictions)

print(f"Best mean CV accuracy: {grid_search.best_score_:.3f}")
print(f"Final test accuracy:   {test_accuracy:.3f}")

Best mean CV accuracy: 0.766
Final test accuracy:   0.775


**Check.** Why should we not now change the hyperparameter grid because we dislike the test accuracy?

<details>
<summary><strong>Answer</strong></summary>

Because the test result would then influence model development. The test set would no longer provide an independent estimate of performance on genuinely unseen data.

</details>

**Takeaway.**

$$
\boxed{\text{search on training data}
\;\longrightarrow\;
\text{select and refit}
\;\longrightarrow\;
\text{evaluate once on test data}}
$$

## Explore and apply

1. Expand the grid by adding `max_depth = [3, 5, 10, None]`. Before running the search, calculate how many cross-validation fits will be required.
2. Give `RandomizedSearchCV` a budget of 10, 30, and 100 iterations. How does increasing the budget affect the best cross-validation score and computation time?
3. Replace the decision tree with a `RandomForestClassifier`. Choose two or three hyperparameters to tune and compare grid search with randomized search.
4. Change the random seed used by the cross-validation splitter. Do the selected hyperparameters remain exactly the same? What does this tell you about model selection from finite data?
5. Explain why tuning on `X_train` and then repeatedly changing the search space after inspecting `X_test` would eventually contaminate the test set.
